<a href="https://colab.research.google.com/github/nalpata/proyecto_ActividadGrado-Riesgos/blob/main/notebooks/01_exploracion_corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Exploración inicial del corpus documental

Este notebook construye el inventario técnico del corpus inicial del proyecto de Actividad de Grado.

**Objetivo:** leer documentos PDF/DOCX, extraer texto, calcular estadísticas básicas, detectar problemas de calidad y preparar evidencia para el pipeline RAG.

## 1. Configuración inicial

Antes de ejecutar, ubica los documentos originales en `data/raw/` dentro del repositorio. Por confidencialidad, estos documentos no se versionan en GitHub.

In [ ]:
from pathlib import Path
import re
import csv

# Librerías para extracción
import fitz  # PyMuPDF
from docx import Document

RAW_DIR = Path('../data/raw')
OUT_DIR = Path('../data/processed/text')
EVAL_DIR = Path('../data/evaluation')

OUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('Carpeta documentos:', RAW_DIR.resolve())
print('Carpeta salida texto:', OUT_DIR.resolve())

## 2. Diccionario inicial de señales de riesgo

Este diccionario no es el modelo final. Es un **baseline explicable** para construir el primer radar de riesgo documental.

In [ ]:
RISK_TERMS = {
    'incumplimiento': ['incumplimiento','incumplimientos','incumplir','incumplido'],
    'retraso_demora': ['retraso','retrasos','demora','demoras','atraso','atrasos'],
    'compromiso_pendiente': ['compromiso','compromisos','pendiente','pendientes'],
    'observacion': ['observación','observaciones','observacion','observaciones'],
    'ans': ['ans','acuerdo de nivel de servicio'],
    'riesgo': ['riesgo','riesgos'],
    'alerta': ['alerta','alertas'],
    'cierre': ['cierre','plan de cierre']
}

RISK_TERMS

## 3. Funciones de extracción y clasificación documental

In [ ]:
def classify_document(filename: str) -> str:
    name = filename.lower()
    if 'acta seguimiento' in name:
        return 'Acta de seguimiento'
    if 'acta review' in name or 'review sprint' in name or 'resultado review' in name:
        return 'Review / acta sprint'
    if 'comunicado' in name:
        return 'Comunicado'
    return 'Otro'


def extract_pdf(path: Path):
    doc = fitz.open(path)
    text = '\n'.join(page.get_text('text') for page in doc)
    return text, len(doc)


def extract_docx(path: Path):
    document = Document(path)
    blocks = [p.text for p in document.paragraphs]
    for table in document.tables:
        for row in table.rows:
            blocks.append(' | '.join(cell.text for cell in row.cells))
    return '\n'.join(blocks), ''


def normalize_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text or '').strip()


def infer_date_from_filename(filename: str) -> str:
    m = re.search(r'(20\d{2})(\d{2})(\d{2})', filename)
    if m:
        return f'{m.group(1)}-{m.group(2)}-{m.group(3)}'
    m = re.search(r'(20\d{2})(\d{2})', filename)
    if m:
        return f'{m.group(1)}-{m.group(2)}'
    return ''


def count_risk_terms(text: str):
    lower = text.lower()
    return {k: sum(lower.count(term) for term in terms) for k, terms in RISK_TERMS.items()}

## 4. Construcción del inventario del corpus

In [ ]:
headers = [
    'id_documento','documento','tipo_documental','formato','fecha_inferida','paginas_pdf',
    'tamano_kb','palabras_extraidas','caracteres_extraidos','estado_extraccion','requiere_ocr',
    'score_senales_riesgo','incumplimiento','retraso_demora','compromiso_pendiente',
    'observacion','ans','riesgo','alerta','cierre','observaciones_limpieza'
]

rows = []
files = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in ['.pdf', '.docx']])

for idx, path in enumerate(files, start=1):
    ext = path.suffix.lower().replace('.', '')
    try:
        if ext == 'pdf':
            text, pages = extract_pdf(path)
        elif ext == 'docx':
            text, pages = extract_docx(path)
        estado = 'OK'
    except Exception as e:
        text, pages = '', ''
        estado = f'ERROR: {str(e)[:80]}'

    clean_text = normalize_spaces(text)
    words = re.findall(r'\b\w+\b', clean_text, flags=re.UNICODE)
    counts = count_risk_terms(clean_text)
    score = sum(counts.values())
    requires_ocr = 'Sí' if ext == 'pdf' and len(words) < 50 else 'No'

    # Guardar texto extraído para etapas posteriores
    output_txt = OUT_DIR / f'{path.stem}.txt'
    output_txt.write_text(text, encoding='utf-8', errors='ignore')

    rows.append([
        f'DOC-{idx:03d}', path.name, classify_document(path.name), ext.upper(),
        infer_date_from_filename(path.name), pages, round(path.stat().st_size/1024, 1),
        len(words), len(clean_text), estado, requires_ocr, score,
        counts['incumplimiento'], counts['retraso_demora'], counts['compromiso_pendiente'],
        counts['observacion'], counts['ans'], counts['riesgo'], counts['alerta'], counts['cierre'],
        'Extracción inicial disponible' if len(words) >= 50 else 'Revisar extracción/OCR'
    ])

len(rows), rows[:2]

## 5. Exportar inventario a CSV

Este archivo puede subirse a GitHub como evidencia del avance del corpus.

In [ ]:
csv_path = EVAL_DIR / 'inventario_corpus.csv'
with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(rows)

print('Inventario guardado en:', csv_path.resolve())

## 6. Resumen para presentación de avance

In [ ]:
total_docs = len(rows)
total_pdf = sum(1 for r in rows if r[3] == 'PDF')
total_docx = sum(1 for r in rows if r[3] == 'DOCX')
total_words = sum(r[7] for r in rows)
total_pdf_pages = sum(int(r[5]) for r in rows if isinstance(r[5], int))
total_risk_score = sum(r[11] for r in rows)

print('Resumen corpus v1')
print('-----------------')
print('Total documentos:', total_docs)
print('PDF:', total_pdf)
print('DOCX:', total_docx)
print('Páginas PDF:', total_pdf_pages)
print('Palabras extraídas:', total_words)
print('Score inicial señales de riesgo:', total_risk_score)

## 6.1 Distribución por tipo documental
from collections import Counter

tipos = [r[2] for r in rows]

Counter(tipos)

## 6.2 Visualizaciòn
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(rows, columns=headers)

df['tipo_documental'].value_counts().plot(kind='bar')

plt.title("Distribución del corpus")
plt.ylabel("Cantidad documentos")
plt.show()

## 6.3 Score de riesgos
df.sort_values(
    'score_senales_riesgo',
    ascending=False
)[[
    'documento',
    'score_senales_riesgo'
]].head(10)

## 7. Próximos pasos

1. Revisar calidad de extracción.
2. Definir reglas de limpieza.
3. Construir chunks con metadatos.
4. Implementar baseline de búsqueda por palabras clave.
5. Comparar contra recuperación semántica en el RAG inicial.